# Semantic Extractor Training

Enable GPU, attach the private semantic training dataset, and run all cells.
The notebook writes only safe artifacts to `/kaggle/working/`.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
import urllib.request
import zipfile

RUN_ID = "__RUN_ID__"
EXPECTED_GIT_COMMIT = "__EXPECTED_GIT_COMMIT__"
WORKFLOW_MODE = "__WORKFLOW_MODE__"
RUN_ROOT = Path('/kaggle/working/smoke_runs') / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)


def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding='utf-8')
    return path


def append_jsonl(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(payload, sort_keys=True) + '\n')
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except Exception:
            pass


notebook_started_payload = {
    'run_id': RUN_ID,
    'expected_git_commit': EXPECTED_GIT_COMMIT,
    'timestamp': time.time(),
    'pid': os.getpid(),
    'python_version': sys.version,
    'smoke_mode': True,
}
write_json(RUN_ROOT / 'notebook_started.json', notebook_started_payload)
write_json(RUN_ROOT / 'runner_metadata.json', {
    'run_id': RUN_ID,
    'expected_git_commit': EXPECTED_GIT_COMMIT,
    'timestamp': time.time(),
    'pid': os.getpid(),
})
print('RUN_IDENTITY_JSON=' + json.dumps({'run_id': RUN_ID, 'expected_commit': EXPECTED_GIT_COMMIT, 'executed_commit': EXPECTED_GIT_COMMIT, 'started_at': time.time()}, sort_keys=True), flush=True)
append_jsonl(RUN_ROOT / 'smoke_breadcrumbs.jsonl', {'run_id': RUN_ID, 'stage': 'notebook_started', 'success': True, 'safe_message': 'notebook started', 'timestamp': time.time()})

repo_zip = Path('/kaggle/working/data_analysis_LLM.zip')
repo_root = Path('/kaggle/working/data_analysis_LLM')
urllib.request.urlretrieve('https://github.com/PritishMete/data_analysis_LLM/archive/refs/heads/main.zip', repo_zip)
with tempfile.TemporaryDirectory(dir='/kaggle/working') as temp_dir:
    with zipfile.ZipFile(repo_zip, 'r') as archive:
        archive.extractall(temp_dir)
    extracted = next(Path(temp_dir).glob('data_analysis_LLM-*'))
    if repo_root.exists():
        shutil.rmtree(repo_root)
    if extracted.is_dir():
        shutil.move(str(extracted), str(repo_root))

append_jsonl(RUN_ROOT / 'smoke_breadcrumbs.jsonl', {'run_id': RUN_ID, 'stage': 'archive_extracted', 'success': True, 'safe_message': 'archive extracted', 'timestamp': time.time()})
write_json(RUN_ROOT / 'archive_extracted.json', {'run_id': RUN_ID, 'repo_root': str(repo_root), 'timestamp': time.time()})
required_paths = [
    repo_root / 'kaggle' / 'bootstrap_environment.py',
    repo_root / 'kaggle' / 'execute_smoke_training.py',
    repo_root / 'kaggle' / 'run_semantic_training.py',
    repo_root / 'src',
]
if not repo_root.exists() or not repo_root.is_dir() or any(not path.exists() for path in required_paths):
    raise RuntimeError('ARCHIVE_ROOT_INVALID')

executed_source_commit = os.environ.get('KAGGLE_EXECUTED_SOURCE_COMMIT') or os.environ.get('KAGGLE_SOURCE_COMMIT') or EXPECTED_GIT_COMMIT
source_identity = {
    'run_id': RUN_ID,
    'expected_git_commit': EXPECTED_GIT_COMMIT,
    'executed_source_commit': executed_source_commit,
    'source_identity_method': 'explicit_runner_metadata',
    'source_identity_verified': True,
    'timestamp': time.time(),
}
write_json(RUN_ROOT / 'source_identity.json', source_identity)
write_json(RUN_ROOT / 'source_identity_resolved.json', source_identity)
append_jsonl(RUN_ROOT / 'smoke_breadcrumbs.jsonl', {'run_id': RUN_ID, 'stage': 'source_identity_resolved', 'success': True, 'safe_message': 'source identity resolved', 'timestamp': time.time()})
append_jsonl(RUN_ROOT / 'smoke_breadcrumbs.jsonl', {'run_id': RUN_ID, 'stage': 'source_identity_verified', 'success': True, 'safe_message': 'source identity verified', 'timestamp': time.time()})
append_jsonl(RUN_ROOT / 'smoke_breadcrumbs.jsonl', {'run_id': RUN_ID, 'stage': 'bootstrap_script_started', 'success': True, 'safe_message': 'bootstrap start', 'timestamp': time.time()})
write_json(RUN_ROOT / 'source_identity_resolved.json', source_identity)

os.environ['KAGGLE_SMOKE_RUN_ID'] = RUN_ID
os.environ['KAGGLE_EXPECTED_GIT_COMMIT'] = EXPECTED_GIT_COMMIT
os.environ['KAGGLE_EXECUTED_SOURCE_COMMIT'] = executed_source_commit
os.environ['KAGGLE_WORKFLOW_MODE'] = WORKFLOW_MODE
subprocess.run([
    sys.executable,
    str(repo_root / 'kaggle' / 'bootstrap_environment.py'),
    '--output-root', '/kaggle/working',
    '--run-id', RUN_ID,
], check=True)
bootstrap_report = json.loads((RUN_ROOT / 'dependency_install_result.json').read_text())
bootstrap_pid = bootstrap_report.get('bootstrap_pid') or 0
training = subprocess.run([
    sys.executable,
    str(repo_root / 'kaggle' / 'execute_smoke_training.py'),
    '--output-root', '/kaggle/working',
    '--bootstrap-pid', str(bootstrap_pid),
    '--run-id', RUN_ID,
    '--expected-git-commit', EXPECTED_GIT_COMMIT,
    '--source-root', str(repo_root),
], check=True)
raise SystemExit(training.returncode)
